In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="darkgrid")
import urllib
import urllib.parse as urlp
import io
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

def get_time_series(start_date,end_date,latitude,longitude,variable):
    """
    Calls the data rods service to get a time series
    """
    base_url = "https://hydro1.gesdisc.eosdis.nasa.gov/daac-bin/access/timeseries.cgi"
    query_parameters = {
        "variable": variable,
        "type": "asc2",
        "location": f"GEOM:POINT({longitude}, {latitude})",
        "startDate": start_date,
        "endDate": end_date,
    }
    full_url = base_url+"?"+ \
         "&".join(["{}={}".format(key,urlp.quote(query_parameters[key])) for key in query_parameters])
    print(full_url)
    iteration = 0
    done = False
    while not done and iteration < 5:
        r=requests.get(full_url)
        if r.status_code == 200:
            done = True
        else:
            iteration +=1
    
    if not done:
        raise Exception(f"Error code {r.status_code} from url {full_url} : {r.text}")
    
    return r.text

def parse_time_series(ts_str):
    """
    Parses the response from data rods.
    """
    lines = ts_str.split("\n")
    parameters = {}
    for line in lines[2:11]:
        key,value = line.split("=")
        parameters[key] = value
    
    
    df = pd.read_table(io.StringIO(ts_str),sep="\t",
                       names=["time","data"],
                       header=10,parse_dates=["time"])
    return parameters, df



In [ ]:

#precipitation rate
df_ts1 = parse_time_series(
        get_time_series(
            start_date="2000-01-01T00", 
            end_date="2025-06-27T23",
            latitude=43.67,
            longitude=-79.54,
            variable="GLDAS2:GLDAS_NOAH025_3H_v2.1:Rainf_tavg"
        )
    )

#Humidity
df_ts2 = parse_time_series(
        get_time_series(
            start_date="2000-01-01T00", 
            end_date="2025-06-27T23",
            latitude=43.67,
            longitude=-79.54,
            variable="GLDAS2:GLDAS_NOAH025_3H_v2.1:Qair_f_inst"
        )
    )

#Pressure
df_ts3 = parse_time_series(
        get_time_series(
            start_date="2000-01-01T00", 
            end_date="2025-06-27T23",
            latitude=43.67,
            longitude=-79.54,
            variable="GLDAS2:GLDAS_NOAH025_3H_v2.1:Psurf_f_inst"
        )
    )

#Downward Shortwave Radiation
df_ts4 = parse_time_series(
        get_time_series(
            start_date="2000-01-01T00", 
            end_date="2025-06-27T23",
            latitude=43.67,
            longitude=-79.54,
            variable="GLDAS2:GLDAS_NOAH025_3H_v2.1:SWdown_f_tavg"
        )
)
#Downward Longwave Radiation
df_ts5 = parse_time_series(
        get_time_series(
            start_date="2000-01-01T00", 
            end_date="2025-06-27T23",
            latitude=43.67,
            longitude=-79.54,
            variable="GLDAS2:GLDAS_NOAH025_3H_v2.1:LWdown_f_tavg"
        )
)

df1 = df_ts1[1].rename({'data': 'Precipitation Rate (mm/s)'},axis='columns')
df2 = df_ts2[1].rename({'data': 'Humidity (g/kg)','time':'t2'},axis='columns')
df3 = df_ts3[1].rename({'data': 'Pressure (Pa)','time':'t3'},axis='columns')
df4 = df_ts4[1].rename({'data': 'Downward Shortwave Radiation (W/m^2)','time':'t4'},axis='columns')
df5 = df_ts5[1].rename({'data': 'Downward Longwave Radiation (W/m^2)','time':'t5'},axis='columns')

combinedDF = pd.concat([df1,df2,df3,df4,df5], axis=1, join="inner")


https://hydro1.gesdisc.eosdis.nasa.gov/daac-bin/access/timeseries.cgi?variable=GLDAS2%3AGLDAS_NOAH025_3H_v2.1%3ARainf_tavg&type=asc2&location=GEOM%3APOINT%28-79.54%2C%2043.67%29&startDate=2000-01-01T00&endDate=2025-06-27T23
https://hydro1.gesdisc.eosdis.nasa.gov/daac-bin/access/timeseries.cgi?variable=GLDAS2%3AGLDAS_NOAH025_3H_v2.1%3AQair_f_inst&type=asc2&location=GEOM%3APOINT%28-79.54%2C%2043.67%29&startDate=2000-01-01T00&endDate=2025-06-27T23
https://hydro1.gesdisc.eosdis.nasa.gov/daac-bin/access/timeseries.cgi?variable=GLDAS2%3AGLDAS_NOAH025_3H_v2.1%3APsurf_f_inst&type=asc2&location=GEOM%3APOINT%28-79.54%2C%2043.67%29&startDate=2000-01-01T00&endDate=2025-06-27T23
https://hydro1.gesdisc.eosdis.nasa.gov/daac-bin/access/timeseries.cgi?variable=GLDAS2%3AGLDAS_NOAH025_3H_v2.1%3ASWdown_f_tavg&type=asc2&location=GEOM%3APOINT%28-79.54%2C%2043.67%29&startDate=2000-01-01T00&endDate=2025-06-27T23
https://hydro1.gesdisc.eosdis.nasa.gov/daac-bin/access/timeseries.cgi?variable=GLDAS2%3AGLDAS_NOAH

## Import Data, Convert into dataframe

### Combine different Raw Data Sets

### Converting Date into MM:DD::YYYY:HH

In [3]:
# drop columns Date 
combinedDF.drop(columns=['t2', 't3', 't4','t5'], inplace=True)
combinedDF.head()

,time,Precipitation Rate (mm/s),Humidity,Pressure (Pa),Downward Shortwave Radiation (W/m^2),Downward Longwave Radiation (W/m^2)
0,2000-01-01 03:00:00,0.000003,0.003743,100311.0,0.00,344.150
1,2000-01-01 06:00:00,0.000001,0.003228,100450.0,0.00,281.701
2,2000-01-01 09:00:00,0.000000,0.003210,100373.0,0.00,253.692
3,2000-01-01 12:00:00,0.000000,0.003310,100242.0,0.00,256.892
4,2000-01-01 15:00:00,0.000002,0.003690,100213.0,71.63,251.769


In [4]:
#split the date column into year, month, day, hour
combinedDF['Year'] = pd.to_datetime(combinedDF['time']).dt.year
combinedDF['Month'] = pd.to_datetime(combinedDF['time']).dt.month
combinedDF['Day'] = pd.to_datetime(combinedDF['time']).dt.day
combinedDF['Hour'] = pd.to_datetime(combinedDF['time']).dt.hour

In [5]:
combinedDF.drop(columns=['time'], inplace=True)
combinedDF.head()

,Precipitation Rate (mm/s),Humidity,Pressure (Pa),Downward Shortwave Radiation (W/m^2),Downward Longwave Radiation (W/m^2),Year,Month,Day,Hour
0,0.000003,0.003743,100311.0,0.00,344.150,2000,1,1,3
1,0.000001,0.003228,100450.0,0.00,281.701,2000,1,1,6
2,0.000000,0.003210,100373.0,0.00,253.692,2000,1,1,9
3,0.000000,0.003310,100242.0,0.00,256.892,2000,1,1,12
4,0.000002,0.003690,100213.0,71.63,251.769,2000,1,1,15


## Adding Latitude and Longitude

In [7]:
#add a column of constant values 
combinedDF['Longitude'] = -79.59
combinedDF['Latitude'] = 43.80
combinedDF.head()

,Precipitation Rate (mm/s),Humidity,Pressure (Pa),Downward Shortwave Radiation (W/m^2),Downward Longwave Radiation (W/m^2),Year,Month,Day,Hour,Longitude,Latitude
0,0.000003,0.003743,100311.0,0.00,344.150,2000,1,1,3,-79.59,43.8
1,0.000001,0.003228,100450.0,0.00,281.701,2000,1,1,6,-79.59,43.8
2,0.000000,0.003210,100373.0,0.00,253.692,2000,1,1,9,-79.59,43.8
3,0.000000,0.003310,100242.0,0.00,256.892,2000,1,1,12,-79.59,43.8
4,0.000002,0.003690,100213.0,71.63,251.769,2000,1,1,15,-79.59,43.8


## Creating the Average Humidity/Pressure Data

In [ ]:
# for each day, calculate 

daily_precip = combinedDF.groupby([ 'Month', 'Day'])['Precipitation Rate (mm/s)'].mean().reset_index()
daily_humidity = combinedDF.groupby([ 'Month', 'Day'])['Humidity'].mean().reset_index()
daily_pressure = combinedDF.groupby([ 'Month', 'Day'])['Pressure (Pa)'].mean().reset_index()
daily_shortwave = combinedDF.groupby([ 'Month', 'Day'])['Downward Shortwave Radiation (W/m^2)'].mean().reset_index()
daily_longwave = combinedDF.groupby([ 'Month', 'Day'])['Downward Longwave Radiation (W/m^2)'].mean().reset_index()

print(daily_pressure)



     Month  Day  Pressure (Pa)
0        1    1   99855.513043
1        1    2  100081.865865
2        1    3  100032.475962
3        1    4   99751.838462
4        1    5   99737.970192
..     ...  ...            ...
361     12   27  100075.739000
362     12   28   99858.259500
363     12   29   99808.578500
364     12   30   99809.335500
365     12   31   99733.306000

[366 rows x 3 columns]


In [10]:
import pickle 

with open('daily_pressure.pkl', 'wb') as f:
    pickle.dump(daily_pressure, f)
    f.close()

with open('daily_humidity.pkl', 'wb') as f:
    pickle.dump(daily_humidity, f)
    f.close()
with open('daily_precip.pkl', 'wb') as f:
    pickle.dump(daily_precip, f)
    f.close()
with open('daily_shortwave.pkl', 'wb') as f:
    pickle.dump(daily_shortwave, f)
    f.close()   
with open('daily_longwave.pkl', 'wb') as f:
    pickle.dump(daily_longwave, f)
    f.close()

### Create Linear Regression Model

In [12]:
X = combinedDF.drop('Precipitation Rate (mm/s)', axis=1)  
y = combinedDF['Precipitation Rate (mm/s)'].values

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size =0.3 , random_state = 89)

In [14]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression() 
linear_model.fit(X_train, y_train)

LinearRegression()

In [15]:
predictions = linear_model.predict(X_test)

linear_model.score(X_test, predictions)

print(linear_model.score(X_test, y_test))

0.05017081752904673


In [16]:
# print significance of feature of linear model
print(linear_model.coef_)

[-1.79530693e-03 -6.10435273e-09 -4.31911889e-08  4.20090181e-07
  7.86752900e-08 -1.70376218e-07  1.01560581e-07  6.42189448e-07
 -1.60308090e-34 -8.01540452e-35]


In [ ]:
daily_precip = combinedDF.groupby([ 'Month', 'Day'])['Precipitation Rate (mm/s)'].mean().reset_index()
daily_humidity = combinedDF.groupby([ 'Month', 'Day'])['Humidity'].mean().reset_index()
daily_pressure = combinedDF.groupby([ 'Month', 'Day'])['Pressure (Pa)'].mean().reset_index()
daily_shortwave = combinedDF.groupby([ 'Month', 'Day'])['Downward Shortwave Radiation (W/m^2)'].mean().reset_index()
daily_longwave = combinedDF.groupby([ 'Month', 'Day'])['Downward Longwave Radiation (W/m^2)'].mean().reset_index()

In [18]:
# create a sample prediction for our matrix

sample = {'Humidity': [daily_humidity['Humidity'][6*30 + 15]],
          'Pressure (Pa)': [daily_pressure['Pressure (Pa)'][6*30 +15]],
          'Downward Shortwave Radiation (W/m^2)': [daily_shortwave['Downward Shortwave Radiation (W/m^2)'][6*30 +15]],
          'Downward Longwave Radiation (W/m^2)': [daily_longwave['Downward Longwave Radiation (W/m^2)'][6*30 +15]],
          'Year': [2023],
          'Month': [6],
          'Day': [15],
          'Hour': [12],
          'Longitude': [-79.59],
          'Latitude': [43.80],
         }

sample_df = pd.DataFrame(sample)

linear_model.predict(sample_df)


array([4.15779722e-05])

In [19]:
# pickle the model 
import pickle
with open('rain_model.pkl', 'wb') as f:
    pickle.dump(linear_model, f)
    f.close()
    

In [37]:
file = open('rain_model.pkl', 'rb')

linear_model = pickle.load(file)

file.close()

linear_model.predict(sample_df)

array([4.23871793e-05])